# Week 1, Day 5: an end-to-end LLM application

Today's idea: put it all together. A request goes through **validate, call the model, parse, guardrails, log**. Each stage is a separate function so each can be tested and a failure can be placed. The project was a recruitment assistant (job description + profiles in, structured evidence-based ratings out). Here is the same pipeline in Python with a stand-in model.

Run the cells from the top, one at a time (Shift+Enter).

In [ ]:
# A STAND-IN model so every cell runs offline. It is NOT a real language model: it follows simple rules and is used only to show the
# code around a model (timing, checks, validation). To use a real model, set MODE = "gemini" and paste a free key from aistudio.google.com.
import json, re, time, urllib.request
MODE = "offline"          # "offline" or "gemini"
GEMINI_KEY = ""           # only for MODE = "gemini"; never share or commit a key

def stand_in(prompt: str) -> str:
    p = prompt.lower()
    if "reply banana" in p: return "BANANA"
    if "sql" in p and "count" in p: return "SELECT COUNT(*) FROM users;"
    m = re.search(r"capital of (\w+)", p)
    if m: return {"france": "Paris", "japan": "Tokyo"}.get(m.group(1), "I am not sure.")
    return "I am not sure."

def ask_model(prompt: str, timeout: float = 20.0):
    """Returns (text, seconds, usage). Same shape for the stand-in and for a real model."""
    t0 = time.perf_counter()
    if MODE == "gemini":
        url = "https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?key=" + GEMINI_KEY
        body = json.dumps({"contents": [{"parts": [{"text": prompt}]}]}).encode()
        req = urllib.request.Request(url, data=body, headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=timeout) as r:
            d = json.load(r)
        text = d["candidates"][0]["content"]["parts"][0]["text"]
        usage = {"input_tokens": d.get("usageMetadata", {}).get("promptTokenCount", 0), "output_tokens": d.get("usageMetadata", {}).get("candidatesTokenCount", 0)}
    else:
        text = stand_in(prompt)
        usage = {"input_tokens": len(prompt.split()), "output_tokens": len(text.split())}   # rough word counts, not real tokens
    return text, time.perf_counter() - t0, usage
print("mode:", MODE)

## Step 1: validate the request first
Refuse bad input before spending a model call.

In [ ]:
class BadRequest(Exception): pass
def validate_request(job, profiles):
    if not 20 <= len(job) <= 6000: raise BadRequest("The job description must be 20 to 6000 characters.")
    if not 1 <= len(profiles) <= 5: raise BadRequest("Send 1 to 5 candidate profiles.")
    if any(not p.strip() for p in profiles): raise BadRequest("A profile is empty.")
    return job.strip(), [p.strip() for p in profiles]

try: validate_request("short", ["x"])
except BadRequest as e: print("refused:", e)

## Step 2: the model call, the parser and the guardrail
The stand-in model returns JSON. The parser refuses anything that does not fit the fixed shape. The guardrail keeps only quotes that are really in the profile.

In [ ]:
def model_analyse(job, profile):                          # STAND-IN for the real model call
    skills = [s for s in ("python", "sql", "docker", "kubernetes") if s in profile.lower() or s == "kubernetes"]   # it "claims" kubernetes even when absent
    return json.dumps({"rating": "strong" if len(skills) > 2 else "unclear", "skills": skills, "evidence": [profile[:40]], "summary": "Stand-in analysis."})

def parse(text):
    try: d = json.loads(text)
    except ValueError: raise BadRequest("The model's answer was not valid JSON.")
    if set(d) != {"rating", "skills", "evidence", "summary"}: raise BadRequest("The answer has the wrong fields.")
    if d["rating"] not in {"strong", "good", "weak", "unclear"}: raise BadRequest("Unknown rating.")
    return d

def guardrail(d, profile):
    text = profile.lower()
    unsupported = [s for s in d["skills"] if s not in text]
    d["skills"] = [s for s in d["skills"] if s in text]
    d["evidence"] = [q for q in d["evidence"] if q.lower() in text]
    d["removed_unsupported"] = unsupported
    return d

## Step 3: the pipeline, with a log
One function runs every stage and writes a log line with timing. Note the log has ids and timings, not the profile text.

In [ ]:
LOG = []
def analyse(job, profiles):
    job, profiles = validate_request(job, profiles)
    results = []
    for i, profile in enumerate(profiles):
        t0 = time.perf_counter()
        d = guardrail(parse(model_analyse(job, profile)), profile)
        LOG.append({"candidate": i, "ms": round((time.perf_counter() - t0) * 1000, 2), "removed": len(d["removed_unsupported"]), "rating": d["rating"]})
        results.append(d)
    return results

JOB = "Backend engineer needed. Python, SQL and Docker required."
out = analyse(JOB, ["Maria. 4 years Python and SQL. Uses Docker daily.", "Tom. Likes cooking."])
for r in out: print(r)
print(LOG)

## Step 4: failure handling and what to practise
Run `analyse` with a bad job description, then with a model that returns broken JSON. Each should give a clear message, not a crash. Try: add a second tool, such as `shortlist(results)`, and a daily call limit per person.

**Where this is in the project:** `AssistantController`, `AssistantService`, `AnalysisPipeline`, `Guardrails.cs`, `docs/evidence/day5.md` (247 tests at the time).